# 🎯 Démo Vertex AI — Prédiction de churn (AutoML Tabular)

**Workshop IST 2025/26 — HEIG-VD.** Plateforme : Google Vertex AI.

On part d'un **CSV de clients** et on obtient un **modèle de churn déployé**, **sans écrire une seule ligne de modèle ML** — uniquement de la configuration.

**Étapes :** 1) Dataset → 2) Entraînement AutoML → 3) Évaluation & importance des features → 4) Prédiction (batch *recommandé*, ou endpoint online).

> ⚠️ **Coût & durée.** L'entraînement AutoML Tabular dure **~1 à 2 h** et coûte **~21 $/node-heure** (≈ 20–40 $ ici). Couvert par le **crédit gratuit (300 $)**. **Lancez l'entraînement AVANT la présentation** et gardez les captures : en live, on montre surtout l'évaluation et la prédiction.


## 0. Installation & authentification *(Colab)*

In [ ]:
# Sur Colab uniquement (déjà présent sur Vertex AI Workbench)
!pip install -q --upgrade google-cloud-aiplatform google-cloud-storage


In [ ]:
# Authentification
import sys
if 'google.colab' in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print('Authentifié via Colab.')
else:
    # En local : `gcloud auth application-default login` au préalable
    print('Hors Colab : utiliser Application Default Credentials.')


## 1. Configuration
Renseignez votre **ID de projet** et un **bucket** Cloud Storage. Région `europe-west6` (**Zurich**) pour garder les données en Suisse.

In [ ]:
PROJECT_ID = 'mon-projet-gcp'        # <-- À MODIFIER
REGION     = 'europe-west6'          # Zurich
BUCKET     = 'gs://mon-bucket-churn' # <-- À MODIFIER (doit exister)

from google.cloud import aiplatform as aip
aip.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET)
print('Vertex AI initialisé sur', PROJECT_ID, REGION)


In [ ]:
# (Optionnel) Activer les APIs et créer le bucket si besoin
# !gcloud services enable aiplatform.googleapis.com storage.googleapis.com --project $PROJECT_ID
# !gsutil mb -l europe-west6 $BUCKET

# Uploader les CSV générés (generate_dataset.py) vers le bucket
!gsutil cp data/clients.csv $BUCKET/clients.csv
!gsutil cp data/clients_a_scorer.csv $BUCKET/clients_a_scorer.csv


## 2. Créer le Dataset tabulaire
Vertex AI peut lire un CSV sur GCS **ou** une table BigQuery (`bq://projet.dataset.table`).

In [ ]:
dataset = aip.TabularDataset.create(
    display_name='clients_churn',
    gcs_source=f'{BUCKET}/clients.csv',
)
print('Dataset créé :', dataset.resource_name)


## 3. Entraîner un modèle AutoML Tabular
On indique simplement la **colonne cible** (`churned`). Vertex teste plusieurs algos, fait le feature engineering et sélectionne le meilleur modèle.

`budget_milli_node_hours=1000` = **1 node-heure** (minimum, suffisant pour la démo).
⏳ **~1–2 h** d'exécution.

In [ ]:
job = aip.AutoMLTabularTrainingJob(
    display_name='churn-automl',
    optimization_prediction_type='classification',
    optimization_objective='maximize-au-prc',  # bon pour classes déséquilibrées
)

model = job.run(
    dataset=dataset,
    target_column='churned',
    budget_milli_node_hours=1000,      # 1 node-heure (~21 $)
    column_transformations=None,        # auto-détection des types
    model_display_name='churn-model',
    disable_early_stopping=False,
)
print('Modèle entraîné :', model.resource_name)


## 4. Évaluation & importance des features
C'est ce qui parle au **métier** : quelles variables expliquent le départ des clients ?

In [ ]:
# Métriques d'évaluation (AUC PR/ROC, précision, rappel, matrice de confusion)
for ev in model.list_model_evaluations():
    m = dict(ev.metrics)
    print('auPrc :', round(m.get('auPrc', 0), 3),
          '| auRoc :', round(m.get('auRoc', 0), 3),
          '| logLoss :', round(m.get('logLoss', 0), 3))

# Importance globale des features (sur la console : onglet 'Feature importance')
# Disponible aussi via l'API d'explication selon le type de modèle.


## 5a. Prédiction BATCH *(recommandée — pas de coût permanent)*
Idéal pour un **scoring mensuel** : on lance le job, il écrit les résultats sur GCS, puis s'arrête.

In [ ]:
batch = model.batch_predict(
    job_display_name='churn-batch',
    gcs_source=f'{BUCKET}/clients_a_scorer.csv',
    gcs_destination_prefix=f'{BUCKET}/predictions/',
    instances_format='csv',
    predictions_format='jsonl',
    machine_type='n1-standard-4',
)
print('Prédictions écrites dans', f'{BUCKET}/predictions/')
# -> chaque ligne contient la proba de churn ; trier pour obtenir la liste 'à risque'.


## 5b. Endpoint ONLINE *(optionnel — temps réel, mais coûteux)*
⚠️ Un endpoint est **facturé 24/7 tant qu'il est déployé**, **même sans trafic**. Toujours **`undeploy`** après la démo.

In [ ]:
endpoint = model.deploy(machine_type='n1-standard-4')

pred = endpoint.predict(instances=[{
    'anciennete_mois': '3', 'type_contrat': 'mensuel',
    'facture_mensuelle': '99.9', 'facture_totale': '299.7',
    'nb_tickets_support': '5', 'moyen_paiement': 'cheque',
    'support_premium': '0', 'age': '29',
}])
print('Prédiction :', pred.predictions)


In [ ]:
# 🔴 IMPORTANT : stopper la facturation de l'endpoint
endpoint.undeploy_all()
endpoint.delete()
print('Endpoint déployé puis nettoyé.')


## 6. Nettoyage & rappel coût
```python
# model.delete(); dataset.delete()
```
**Récapitulatif coût (scoring mensuel) :** endpoint 24/7 ≈ **CHF ~195/mois** vs **batch ≈ CHF ~60/mois**.
👉 En l'absence de besoin temps réel, **le batch divise la facture par ~3**.
